# just_drop vs just_drop_hal PPS

对比 **XDP ingress drop**（`examples/just_drop`）与 **HAL 应用层 drop**（`examples/just_drop_hal`）在同等 UDP 洪泛下的 DUT `drop` 吞吐。

数据一律经 `utils.db` **现查 Postgres**（`benchmark/.env`），不落本地结果缓存。方法与探针命令见仓库 [`README.md`](../README.md)。


## 环境与连接

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from utils.db import dsn_summary
from utils import pps

print(dsn_summary())
pps.latest_runs()


## 指标

| 符号 | 定义 |
|------|------|
| `overall_pps` | `(drop_end − drop_start) / blast.elapsed_secs` |
| `interval_pps` | 相邻采样差分 / Δt |
| `host_pps` | TR3000 发送计数 / blast 时长 |

窗口 **40s**、采样 **1000ms**、每臂 **3** 轮。XDP 要求 `net_rx` 不变；HAL 在 `drop on` 下亦不增 `net_rx`（不 ACK）。


## 结果摘要（库内最新 run）

In [ ]:
summary = pps.path_summary()
summary


In [ ]:
rounds = pps.rounds_for_latest()
rounds


In [ ]:
# 比值 XDP / HAL（overall_pps mean）
s = summary.set_index("path")
if {"hal", "xdp"}.issubset(s.index):
    ratio = float(s.loc["xdp", "overall_pps_mean"]) / float(s.loc["hal", "overall_pps_mean"])
    print(f"XDP / HAL ≈ {ratio:.2f}×")


## 区间 pps 时序

In [ ]:
import matplotlib.pyplot as plt

samples = pps.samples_for_latest()
fig, ax = plt.subplots(figsize=(9, 4.5))
for path, color in (("hal", "#1f77b4"), ("xdp", "#d62728")):
    sub = samples[(samples["path"] == path) & (samples["t_s"] > 0)]
    # mean interval pps across rounds, bucketed by floor(t)
    g = sub.assign(bucket=sub["t_s"].astype(int)).groupby("bucket")["interval_pps"].mean()
    ax.plot(g.index, g.values, label=path, color=color, linewidth=1.5)
ax.set_xlabel("t (s)")
ax.set_ylabel("interval pps")
ax.set_title("DUT drop interval pps (live from Postgres)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## overall pps 柱状（mean ± std）

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
paths = list(summary["path"])
means = summary["overall_pps_mean"].astype(float)
stds = summary["overall_pps_std"].fillna(0).astype(float)
colors = ["#1f77b4" if p == "hal" else "#d62728" for p in paths]
xs = range(len(paths))
ax.bar(xs, means, yerr=stds, color=colors, capsize=6, width=0.55)
ax.set_xticks(list(xs))
ax.set_xticklabels(paths)
ax.set_ylabel("overall pps")
ax.set_title("DUT drop overall pps (mean ± std)")
ax.grid(True, axis="y", alpha=0.3)
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s, f"{m:.0f}", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
plt.show()


## 解读

- Host 发送约 **9k pps**；XDP 臂 `overall_pps ≈ host_pps`，ingress drop 能跟上 blast。
- HAL 臂约 **2.7k pps**（约为 host 的 30%）：包已进入 smoltcp/UDP 再记 drop，栈路径是瓶颈。
- 本对比度量的是 **可观测 drop 计数吞吐**，不是同层（ingress vs 应用）的公平微基准。
